# VAVE AI: Simplified Stable Diffusion 3 Medium Training
This notebook handles dataset extraction, simplified LoRA fine-tuning, testing, and validation of custom automotive components.

### 1. Setup Environment & HuggingFace Login
Ensure you are using a GPU Runtime in Colab (T4, L4, or A100). Use your `HF_TOKEN` from HuggingFace to login.

In [ ]:
!pip install -q -U diffusers transformers accelerate peft huggingface_hub

import os
from huggingface_hub import login

# NOTE: Replace with your actual HuggingFace Read Token!
# Make sure you accepted the License on the model page first.
login(token="YOUR_HF_TOKEN_HERE")

### 2. Extract Dataset
Upload your `dataset.zip` file using the folder icon on the left, then run this block to unzip it.

In [ ]:
!unzip -q dataset.zip -d dataset/

### 3. Simplified SD3 LoRA Fine-Tuning
We use the official diffusers dreambooth script with minimal essential parameters.

In [ ]:
!git clone https://github.com/huggingface/diffusers.git
%cd diffusers/examples/dreambooth

# Install the specific training dependencies
!pip install -r requirements_sd3.txt

!accelerate launch train_dreambooth_lora_sd3.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-3-medium-diffusers" \
  --dataset_name="/content/dataset" \
  --instance_prompt="A realistic automotive engineering photo" \
  --output_dir="/content/sd3-vave-lora" \
  --mixed_precision="fp16" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_steps=500

### 4. Download Weights to PC
Zip and download the resulting fine-tuned weights.

In [ ]:
!zip -r sd3-vave-lora.zip /content/sd3-vave-lora

from google.colab import files
files.download("sd3-vave-lora.zip")

### 5. Test Inference
Test the generated LoRA weights visually.

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline
from IPython.display import display

pipeline = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16,
    text_encoder_3=None,
    tokenizer_3=None
)
pipeline.load_lora_weights("/content/sd3-vave-lora")
pipeline.to("cuda")

prompt = "A realistic automotive engineering photo showing a Brake Caliper. Cost reduction idea: use cast aluminum."
image = pipeline(prompt, num_inference_steps=28, guidance_scale=7.0).images[0]
image.save("/content/test_output.png")
display(image)